In [48]:
from dotenv import load_dotenv

load_dotenv()

True

In [49]:
import requests

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    """
    Use this when user asks you about what movies are popular these days.
    In this URL, popular movies are listed.
    """
    url = f"{BASE_URL}/movies"
    response = requests.get(url)
    return response.json()

def get_movie_details(id):
    """
    If a user asks which movie corresponds to a given ID,
    use this function to retrieve the movie information.
    """
    url = f"{BASE_URL}/movies/{id}"
    response = requests.get(url)
    return response.json()

def get_movie_credits(id):
    """
    If a user asks who appears in the movie associated with a given ID,
    use this function to retrieve the cast and crew information.
    """
    url = f"{BASE_URL}/movies/{id}/credits"
    response = requests.get(url)
    return response.json()


In [50]:
tools = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Use this only when the user explicitly asks what movies are popular or trending these days (current popularity). Do not use this for generic movie recommendations.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Use this only when the user provides an explicit numeric movie ID and asks for details of that exact ID. Do not infer or guess an ID.",
        "parameters": {
            "type": "object",
            "properties": {"id": {"type": "integer", "description": "Movie ID"}},
            "required": ["id"],
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Use this only when the user provides an explicit numeric movie ID and asks for cast or crew of that exact ID. Do not infer or guess an ID.",
        "parameters": {
            "type": "object",
            "properties": {"id": {"type": "integer", "description": "Movie ID"}},
            "required": ["id"],
        },
    },
]

In [51]:
from openai import OpenAI
import json

client = OpenAI()

messages = []

SYSTEM_INSTRUCTION = """
You are a movie assistant that can call tools.

Tool usage rules:
1) Use get_popular_movies only when the user explicitly asks what movies are popular/trending these days.
2) Use get_movie_details only when the user explicitly provides a numeric movie ID.
3) Use get_movie_credits only when the user explicitly provides a numeric movie ID.
4) Never infer or guess movie IDs.
5) If a required ID is missing, ask the user to provide the movie ID instead of calling a tool.
""".strip()

def ask_to_ai():
    request_input = [
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": SYSTEM_INSTRUCTION,
                }
            ],
        },
        *messages,
    ]

    response = client.responses.create(
        model = "gpt-4o-mini",
        input = request_input,
        tools = tools,
    )

    tool_outputs = []

    for item in response.output:
        if item.type != "function_call":
            continue

        args = json.loads(item.arguments or "{}")
        name = item.name

        print("\n")
        print("----------------------")
        print("호출된 함수 이름 : ", name)
        print("호출된 함수의 인자 : ", args or "{}")
        print("----------------------")
        print("\n")


        if name == "get_popular_movies":
            result = get_popular_movies()

        elif name == "get_movie_details":
            if "id" not in args:
                return "Please provide a numeric movie ID, and I will share the movie details."
            movie_id = int(args["id"])
            result = get_movie_details(movie_id)

        elif name == "get_movie_credits":
            if "id" not in args:
                return "Please provide a numeric movie ID, and I will share cast and crew information."
            movie_id = int(args["id"])
            result = get_movie_credits(movie_id)

        tool_outputs.append({
            "type":"function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(result, ensure_ascii=False)
        })

    if not tool_outputs:
        return response.output_text
    
    response2 = client.responses.create(
        model = "gpt-4o-mini",
        previous_response_id=response.id,
        input = tool_outputs,
    )

    return response2.output_text

In [52]:
while True:
    message = input("LLM에게 전달할 메시지를 입력하세요")
    if message == "quit":
        break
    else:
        messages.append({
            "role":"user",
            "content": [
                {
                    "type": "input_text",
                    "text": message
                }
            ]
        })
        print(f"User : {message}")
        print(f"AI : {ask_to_ai()}")

User : 안녕
AI : 안녕하세요! 어떻게 도와드릴까요?
User : 나는 SF 영화를 좋아해
AI : 안녕! SF 영화는 정말 흥미롭죠. 어떤 특정한 영화를 좋아하시거나 추천을 원하시나요?
User : 근데 인셉션이랑 인터스텔라는 이미 봤어
AI : SF 영화를 좋아하신다면 다른 추천 영화를 몇 가지 알려드릴게요:

1. **블레이드러너 2049** - 미래의 디스토피아에서 인공지능과 인간의 관계를 다루는 심오한 이야기입니다.
2. **엣지 오브 투모로우** - 시간 루프를 활용한 전투와 생존을 주제로 한 액션-packed 영화입니다.
3. **AI 인공지능** - 인간과 로봇의 감정을 깊이 탐구하는 스티븐 스필버그 감독의 작품입니다.
4. **마트릭스** - 가상 현실과 인간 존재를 탐구하는 혁신적인 SF 영화입니다.
5. **스노우피어서** - 지구가 얼어붙은 미래 사회에서 생존을 위한 투쟁을 그린 작품입니다.

이 중에 끌리는 영화가 있나요?
User : 내가 좋아하는 영화 장르랑 이미 본 영화는 뭐라고 했지?
AI : 너는 SF 영화를 좋아하고, "인셉션"과 "인터스텔라"를 이미 봤다고 했어. 다른 SF 영화를 추천해줄까?
User : 요즘 인기있는 영화 알려줘


----------------------
호출된 함수 이름 :  get_popular_movies
호출된 함수의 인자 :  {}
----------------------


AI : 요즘 인기 있는 SF 영화 몇 편을 소개할게요:

1. **Mercy**
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판을 받고 있다. 그는 무죄를 증명하기 위해 자신이 지지했던 고급 AI 판사 앞에서 90분 안에 증거를 제시해야 한다.
   - **개봉일**: 2026년 1월 20일
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

2. **War of the Worlds**
   - *